# DPsurv on KIRC — End-to-End Pipeline

This notebook demonstrates the full **DPsurv** workflow on the **TCGA-KIRC** (Kidney Renal Clear Cell Carcinoma) dataset.

**Pipeline:**
1. Load pre-computed PANTHER GMM embeddings (π, μ, Σ) for each KIRC slide
2. Run nested cross-validation:
   - Inner loop: select K ∈ {1,2,3,4} via 15% held-out validation (early stopping on C-index)
   - Outer loop: retrain on full outer-train fold, evaluate on test
3. Aggregate 5-fold metrics (C-index, C-index_td, IBS, NBLL)
4. Visualize training curves and per-fold results

**Input pathway:** GMM embeddings (not raw patches) → DPsurv (evidential mixture model)

> Adjust the paths in **Cell 0** to match your local setup.

In [ ]:
# ─── Cell 0: Configuration ───────────────────────────────────────────────────
import sys
from pathlib import Path

# Add repo root to Python path
REPO_ROOT = Path("../").resolve()   # DPsurv/ directory
sys.path.insert(0, str(REPO_ROOT))

DATASET        = "KIRC"
SEED           = 42
DEVICE         = "cuda"           # or "cpu"
SPLITS_ROOT    = REPO_ROOT / "data" / "splits"
RESULTS_DIR    = REPO_ROOT / "results"

EMBEDDING_FNAME = (
    "extracted-vit_large_patch16_224.dinov2.uni_mass100k_"
    "PANTHER_embeddings_proto_16_allcat_em_1_eps_1.0_tau_1.0_tokenized.pkl"
)

# DPsurv hyperparameters (defaults match the paper)
HPARAMS = dict(
    k_values           = [1, 2, 3, 4],
    inner_val_fraction = 0.15,
    n_label_bins       = 8,
    max_epochs         = 50,
    min_epochs         = 5,
    patience           = 5,
    batch_size         = 32,
    eval_batch_size    = 1000,
    lr                 = 1e-4,
    weight_decay       = 2e-4,
    weight             = 0.5,
    alpha              = 0.5,
    xi                 = 0.0,
    rho                = 0.0,
    warmup_ratio       = 0.1,
    eta_min            = 1e-6,
    gamma_scale        = 0.5,
    prob_thresh        = 0.01,
    kmeans_nstart      = 100,
    num_workers        = 0,
)

print(f"Repo root : {REPO_ROOT}")
print(f"Splits    : {SPLITS_ROOT}")
print(f"Results   : {RESULTS_DIR}")

In [ ]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────────
import gc
import json
import pickle
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

from downstream.dpsurv.models import (
    ENNreg_init_cosine,
    mixture_ENNreg_new,
    mixture_input_dim_from_mean_dim,
)
from downstream.dpsurv.losses import (
    Mixture_Evidential_nll_Loss,
    evaluate_nll_batch_survival,
)
from downstream.dpsurv.data import GMMEmbeddingDataset, build_df, collate_flat

print("Imports OK")

In [ ]:
# ─── Cell 2: Helper functions (mirrors trainer/train_dpsurv.py) ──────────────
DATASET_ALIASES = {
    "BLCA": "TCGA_BLCA", "BRCA": "TCGA_BRCA",
    "KIRC": "TCGA_KIRC", "LUAD": "TCGA_LUAD", "UCEC": "TCGA_UCEC",
}


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def fold_dir(dataset, fold):
    return SPLITS_ROOT / f"{DATASET_ALIASES[dataset]}_overall_survival_k={fold}"

def load_fold_frames(dataset, fold):
    fdir = fold_dir(dataset, fold)
    embed_path = fdir / "embeddings" / EMBEDDING_FNAME
    with open(embed_path, 'rb') as f:
        emb = pickle.load(f)
    train_df = pd.read_csv(fdir / "train.csv")
    test_df  = pd.read_csv(fdir / "test.csv")
    train_df = build_df(emb["train"], train_df).drop_duplicates('case_id').dropna()
    test_df  = build_df(emb["test"],  test_df).drop_duplicates('case_id').dropna()
    return train_df, test_df

def compute_qbins(df, n_bins):
    uncensored = df[df['dss_censorship'].astype(float) == 0]
    src = uncensored if len(uncensored) >= 2 else df
    for q in range(n_bins, 0, -1):
        try:
            _, bins = pd.qcut(src['dss_survival_days'].astype(float),
                              q=q, retbins=True, labels=False, duplicates='drop')
            bins = np.unique(bins.astype(np.float32))
            if len(bins) >= 2:
                return bins
        except Exception:
            continue
    return np.array([0., 1e6], dtype=np.float32)

def make_loader(ds, batch_size, shuffle):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_flat, num_workers=HPARAMS['num_workers'])

def init_prototypes(train_df, k_value):
    prob_np = np.stack(train_df['prob'].to_numpy())
    mean_np = np.stack(train_df['mean'].to_numpy())
    log_surv = torch.log(torch.tensor(train_df['dss_survival_days'].values, dtype=torch.float32))
    n_comp = prob_np.shape[1]
    return [
        ENNreg_init_cosine(
            torch.tensor(prob_np[:, i], dtype=torch.float32),
            torch.tensor(mean_np[:, i, :], dtype=torch.float32),
            log_surv, k_value,
            nstart=HPARAMS['kmeans_nstart'],
            c=HPARAMS['gamma_scale'],
            prob_thresh=HPARAMS['prob_thresh'],
        )
        for i in range(n_comp)
    ]

def build_model(train_df, k_value, device):
    protos = init_prototypes(train_df, k_value)
    mean0  = np.asarray(train_df['mean'].iloc[0])
    input_dim = mixture_input_dim_from_mean_dim(mean0.shape[1])
    return mixture_ENNreg_new(input_dim=input_dim, prototype_list=protos, num_models=len(protos)).to(device)

def build_loss(qbins):
    return Mixture_Evidential_nll_Loss(
        qbins=qbins,
        alpha=HPARAMS['alpha'],
        lambd=HPARAMS['weight'],
        xi=HPARAMS['xi'],
        rho=HPARAMS['rho'],
    )

def build_optimizer(model):
    beta_p, w_p, other_p = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'beta' in name: beta_p.append(p)
        elif 'w' in name:  w_p.append(p)
        else: other_p.append(p)
    return torch.optim.AdamW(
        [{'params': beta_p}, {'params': w_p}, {'params': other_p}],
        lr=HPARAMS['lr'], weight_decay=HPARAMS['weight_decay'],
    )

def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device):
    model.train()
    total = 0.0
    for batch in loader:
        prob = batch['prob'].to(device)
        mean = batch['mean'].to(device)
        cov  = batch['cov'].to(device)
        y    = batch['label'].to(device)
        c    = batch['censorship'].to(device)
        optimizer.zero_grad()
        out = model(torch.cat([prob, mean.reshape(mean.shape[0], -1),
                               cov.reshape(cov.shape[0], -1)], dim=1), prob)
        loss_dict = loss_fn(out, y, c, prob)
        loss_dict['loss'].backward()
        optimizer.step()
        if scheduler: scheduler.step()
        total += loss_dict['loss'].item()
    return total / max(len(loader), 1)

print("Helpers defined")

In [ ]:
# ─── Cell 3: Load KIRC splits (show fold 0 as example) ───────────────────────
set_seed(SEED)
device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

outer_train_df, test_df = load_fold_frames(DATASET, fold=0)
print(f"Fold 0  — outer_train: {len(outer_train_df)}  test: {len(test_df)}")
print(outer_train_df[['case_id', 'dss_survival_days', 'dss_censorship']].head())

In [ ]:
# ─── Cell 4: Inner K selection (15% held-out validation) ─────────────────────
# Shows the inner-loop logic for fold 0

inner_train_df, inner_val_df = train_test_split(
    outer_train_df, test_size=HPARAMS['inner_val_fraction'],
    random_state=SEED, shuffle=True,
)
print(f"Inner train: {len(inner_train_df)}  val: {len(inner_val_df)}")

qbins = compute_qbins(inner_train_df, HPARAMS['n_label_bins'])
print(f"Quantile bins: {qbins}")

train_ds = GMMEmbeddingDataset(inner_train_df, qbins=qbins)
val_ds   = GMMEmbeddingDataset(inner_val_df,   qbins=qbins)

k_results = {}

for k in tqdm(HPARAMS['k_values'], desc='K grid search'):
    set_seed(SEED)
    model    = build_model(inner_train_df, k, device)
    loss_fn  = build_loss(qbins)
    optimizer = build_optimizer(model)

    best_c  = -np.inf
    wait    = 0
    history = []

    for epoch in range(1, HPARAMS['max_epochs'] + 1):
        tr_loss = train_one_epoch(
            model, make_loader(train_ds, HPARAMS['batch_size'], True),
            loss_fn, optimizer, None, device
        )
        with torch.no_grad():
            metrics = evaluate_nll_batch_survival(
                model, make_loader(val_ds, HPARAMS['eval_batch_size'], False),
                device, qbins, weight=HPARAMS['weight']
            )
        history.append({'epoch': epoch, 'tr_loss': tr_loss, **metrics})

        if epoch >= HPARAMS['min_epochs'] and metrics['c_index'] > best_c:
            best_c  = metrics['c_index']
            best_ep = epoch
            wait    = 0
        elif epoch >= HPARAMS['min_epochs']:
            wait += 1
            if wait >= HPARAMS['patience']:
                break

    k_results[k] = {'best_c_index': best_c, 'best_epoch': best_ep, 'history': history}
    print(f"  K={k}: best val C-index={best_c:.4f} at epoch {best_ep}")
    del model; gc.collect(); torch.cuda.empty_cache()

best_k  = max(k_results, key=lambda k: k_results[k]['best_c_index'])
best_ep = k_results[best_k]['best_epoch']
print(f"\nSelected K={best_k}  (epoch={best_ep}, val C-index={k_results[best_k]['best_c_index']:.4f})")

In [ ]:
# ─── Cell 5: Retrain on full outer-train and evaluate on test (fold 0) ────────

set_seed(SEED)
qbins_final = compute_qbins(outer_train_df, HPARAMS['n_label_bins'])
train_ds_f  = GMMEmbeddingDataset(outer_train_df, qbins=qbins_final)
test_ds_f   = GMMEmbeddingDataset(test_df,        qbins=qbins_final)

model_f    = build_model(outer_train_df, best_k, device)
loss_fn_f  = build_loss(qbins_final)
optimizer_f = build_optimizer(model_f)

final_history = []
for epoch in tqdm(range(1, best_ep + 1), desc='Final training'):
    tr_loss = train_one_epoch(
        model_f, make_loader(train_ds_f, HPARAMS['batch_size'], True),
        loss_fn_f, optimizer_f, None, device
    )
    final_history.append({'epoch': epoch, 'tr_loss': tr_loss})

test_metrics = evaluate_nll_batch_survival(
    model_f, make_loader(test_ds_f, HPARAMS['eval_batch_size'], False),
    device, qbins_final, weight=HPARAMS['weight']
)

print(f"\nFold 0 Test Results (K={best_k}, epochs={best_ep}):")
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

In [ ]:
# ─── Cell 6: Run all 5 folds and aggregate metrics ───────────────────────────
# (Uses the same K selection + retraining logic as above)
# NOTE: This cell calls trainer/train_dpsurv.py programmatically.
#       Alternatively, run: bash scripts/run_dpsurv.sh KIRC 1

import subprocess, sys

cmd = [
    sys.executable,
    str(REPO_ROOT / 'trainer' / 'train_dpsurv.py'),
    '--datasets', DATASET,
    '--device',   DEVICE,
    '--results_dir', str(RESULTS_DIR),
]

print("Running:", ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_ROOT))
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

In [ ]:
# ─── Cell 7: Load and display aggregated 5-fold results ─────────────────────
summary_path = RESULTS_DIR / DATASET / 'summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"DPsurv on {DATASET} — 5-fold aggregated results:")
    print(f"  C-index    : {summary['c_index_mean']:.4f} ± {summary['c_index_std']:.4f}")
    print(f"  C-index_td : {summary['c_index_td_mean']:.4f} ± {summary['c_index_td_std']:.4f}")
    print(f"  IBS        : {summary['ibs_mean']:.4f} ± {summary['ibs_std']:.4f}")
    print(f"  NBLL       : {summary['nbll_mean']:.4f} ± {summary['nbll_std']:.4f}")
else:
    print(f"Summary not found at {summary_path}. Run Cell 6 first.")

In [ ]:
# ─── Cell 8: Plot training curve (fold 0) ────────────────────────────────────
epochs_f = [h['epoch']   for h in final_history]
losses_f = [h['tr_loss'] for h in final_history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
axes[0].plot(epochs_f, losses_f, color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title(f'DPsurv on KIRC — Fold 0 Training (K={best_k})')
axes[0].grid(alpha=0.3)

# Inner K selection comparison
for k, res in k_results.items():
    hist = res['history']
    ep   = [h['epoch']   for h in hist]
    ci   = [h['c_index'] for h in hist]
    axes[1].plot(ep, ci, label=f'K={k}', linewidth=1.5)
axes[1].axvline(best_ep, color='k', linestyle='--', alpha=0.5, label=f'Selected ep={best_ep}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val C-index')
axes[1].set_title('Inner K Selection — Validation C-index')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / f'{DATASET}_fold0_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved training curve.")

In [ ]:
# ─── Cell 9: Per-fold metrics table (after running all folds via Cell 6) ────
metrics_csv = RESULTS_DIR / DATASET / 'fold_metrics.csv'
if metrics_csv.exists():
    df_m = pd.read_csv(metrics_csv)
    display(df_m)
    agg = df_m[['c_index', 'c_index_td', 'ibs', 'nbll']].agg(['mean', 'std'])
    print("\nAggregated:")
    display(agg.round(4))
else:
    print(f"Per-fold CSV not found at {metrics_csv}. Run Cell 6 first.")